# Notebook 7 — Visualisation complète du projet IAQ

**Détection d'anomalies de pollution de l'air intérieur par graphe de connaissances et GNN**

Ce notebook lit les **fichiers de résultats réels** produits par les notebooks 0 à 6 (et par les
scripts de ré-analyse de seuil) dans le dossier `processed/`, puis génère **toutes les figures**
du rapport. Chaque figure est :

- construite à partir des données réelles (aucune valeur inventée) ;
- affichée dans le notebook **et** enregistrée en PNG dans le dossier `figures/`, prête à être
  insérée dans le rapport.

Les figures sont organisées en six groupes :

| Groupe | Contenu |
|---|---|
| **A** | Données (séries temporelles, événements, corrélations, états) |
| **B** | Ontologie (résumé structurel, catégories sémantiques) |
| **C** | Graphe GNN (topologie à 30 nœuds, degrés, relations) |
| **D** | **Détection** (queue lourde, seuils, l'histoire 0/3 → 3/3) |
| **E** | Prédiction (comparaison GNN vs séquentiel, R² par polluant) |
| **F** | Synthèse (tableau de bord final) |


In [ ]:
# === Configuration et chargement ===
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.dates as mdates
import networkx as nx
warnings.filterwarnings("ignore")

# --- Localiser le dossier processed/ (à côté de ce notebook) ---
def find_processed():
    for c in [Path("processed"), Path("../processed"), Path(".")]:
        if (c / "config.json").exists():
            return c
        if (c / "processed" / "config.json").exists():
            return c / "processed"
    raise FileNotFoundError("Dossier 'processed' introuvable - placez ce notebook à la racine du projet.")

PROC  = find_processed()
GRAPH = PROC / "graph"
ONTO  = PROC / "ontology"
FIG   = Path("figures"); FIG.mkdir(exist_ok=True)
print("processed :", PROC.resolve())
print("figures   :", FIG.resolve())

# --- Style global ---
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
})
CAT_COLORS = {
    "Pollutant": "#d64545", "ParticleMatter": "#e08a3c", "ParticleCount": "#f0c419",
    "EnvironmentalCondition": "#2e86c1", "DynamicCondition": "#16a085",
    "AcousticCondition": "#8e44ad", "AssessmentMetric": "#7f8c8d",
    "MeasurementProperty": "#34495e",
}
EVENT_COLORS = {"fire": "#e74c3c", "dust_humidity": "#f39c12", "power_outage": "#2c3e50"}
EVENT_FR = {"fire": "Incendie", "dust_humidity": "Erreur poussière/humidité",
            "power_outage": "Coupure de courant", "normal": "Normal"}

def load_json(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

# --- Charger les métadonnées et résultats réels ---
cfg      = load_json(PROC / "config.json")
onto_rep = load_json(ONTO / "ontology_report.json")
gman     = load_json(GRAPH / "graph_manifest.json")
events   = cfg["events"]; cat = cfg["category"]; units = cfg["units"]

def save_show(fig, name):
    """Enregistre la figure en PNG dans figures/ puis l'affiche dans le notebook."""
    fig.savefig(FIG / name)
    plt.show()
    print("figure enregistrée :", FIG / name)

print("\nFichiers de résultats détectés :")
for f in ["detection_robust_summary.json", "detection_robust.csv", "detection_reeval.csv",
          "nb05_pred_metrics.csv", "nb05_pred_r2.csv", "nb06_final_summary.json",
          "recon_errors.npz", "graph_manifest.json"]:
    print(("  OK  " if (GRAPH / f).exists() else "  --  ") + f)


---
## Groupe A — Les données

Deux déploiements réels en Allemagne (2023) : un **laboratoire** (22 mars → 6 juin, ~54 700 lignes
après ré-échantillonnage à 2 min) et un **appartement** (24 juin → 11 juillet). Trois événements
anormaux connus servent de vérité terrain :

- **Coupure de courant** (laboratoire, 17–19 avril) : apparaît comme un **trou de ~44 h** dans les
  données → anomalie *structurelle*.
- **Incendie** (laboratoire, 21 mai) : grand incendie à ~9–10 km → **élévation subtile du NO₂**,
  réponse PM faible → anomalie *de valeur*, difficile.
- **Erreur poussière/humidité** (appartement, 9 juillet) : artefact de mesure PM2.5 lors d'une
  hausse d'humidité → **pic PM2.5 fort**.

Les figures A1–A4 montrent ces événements ; A5 les corrélations entre variables ; A6 la répartition
des états de qualité d'air (selon les seuils OMS / ASHRAE / EN 16798-1).

In [ ]:
# === A1-A2 : séries temporelles avec événements ===
lab = pd.read_csv(PROC / "laboratory_clean.csv", parse_dates=["timestamp"])
apt = pd.read_csv(PROC / "apartment_clean.csv", parse_dates=["timestamp"])

def ev_span(name):
    e = next(x for x in events if x["name"] == name)
    return pd.Timestamp(e["start"]), pd.Timestamp(e["end"])

# A1 - laboratoire
cols = ["co2", "no2", "pm2_5", "tvoc"]
fig, axes = plt.subplots(len(cols), 1, figsize=(11, 9), sharex=True)
po_s, po_e = ev_span("power_outage"); fi_s, fi_e = ev_span("fire")
for ax, c in zip(axes, cols):
    ax.plot(lab["timestamp"], lab[c], lw=0.6, color="#2c3e50")
    ax.axvspan(po_s, po_e, color=EVENT_COLORS["power_outage"], alpha=0.18)
    ax.axvspan(fi_s, fi_e, color=EVENT_COLORS["fire"], alpha=0.30)
    ax.set_ylabel(f"{c}\n({units.get(c,'')})")
axes[0].set_title("Laboratoire - séries temporelles avec événements\n"
                  "(gris = coupure courant ~44 h, rouge = incendie 21 mai)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b")); axes[-1].set_xlabel("Date (2023)")
save_show(fig, "A1_labo_series_temporelles.png")

# A2 - appartement
cols = ["pm2_5", "humidity", "pm10"]
fig, axes = plt.subplots(len(cols), 1, figsize=(11, 7), sharex=True)
d_s, d_e = ev_span("dust_humidity")
for ax, c in zip(axes, cols):
    ax.plot(apt["timestamp"], apt[c], lw=0.6, color="#2c3e50")
    ax.axvspan(d_s, d_e, color=EVENT_COLORS["dust_humidity"], alpha=0.30)
    ax.set_ylabel(f"{c}\n({units.get(c,'')})")
axes[0].set_title("Appartement - séries temporelles\n(orange = erreur poussière/humidité, 9 juillet)")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b")); axes[-1].set_xlabel("Date (2023)")
save_show(fig, "A2_appartement_series_temporelles.png")


In [ ]:
# === A3-A4 : zooms sur les deux événements de valeur ===
# A3 - incendie (NO2)
fi_s, fi_e = ev_span("fire"); pad = pd.Timedelta(days=2)
sub = lab[(lab["timestamp"] >= fi_s - pad) & (lab["timestamp"] <= fi_e + pad)]
normal_no2 = lab.loc[lab["event"] == "normal", "no2"].mean()
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(sub["timestamp"], sub["no2"], color=EVENT_COLORS["fire"], lw=1.2, label="NO2")
ax.axvspan(fi_s, fi_e, color=EVENT_COLORS["fire"], alpha=0.18, label="Fenêtre incendie")
ax.axhline(normal_no2, ls="--", color="#2c3e50", lw=1, label=f"Moyenne normale ≈ {normal_no2:.1f}")
ax.set_title("Zoom incendie (21 mai) - élévation subtile du NO2")
ax.set_ylabel(f"NO2 ({units.get('no2','µg/m³')})"); ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Hh"))
save_show(fig, "A3_zoom_incendie_no2.png")

# A4 - poussière/humidité (PM2.5 + humidité)
d_s, d_e = ev_span("dust_humidity")
sub = apt[(apt["timestamp"] >= d_s - pad) & (apt["timestamp"] <= d_e + pad)]
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.plot(sub["timestamp"], sub["pm2_5"], color=EVENT_COLORS["dust_humidity"], lw=1.2)
ax1.set_ylabel("PM2.5 (µg/m³)", color=EVENT_COLORS["dust_humidity"])
ax1.axvspan(d_s, d_e, color=EVENT_COLORS["dust_humidity"], alpha=0.15)
ax2 = ax1.twinx(); ax2.plot(sub["timestamp"], sub["humidity"], color="#2e86c1", lw=1.0, alpha=0.7)
ax2.set_ylabel("Humidité (%)", color="#2e86c1"); ax2.grid(False)
ax1.set_title("Zoom erreur poussière/humidité (9 juillet)\nartefact PM2.5 lors d'une hausse d'humidité")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
save_show(fig, "A4_zoom_poussiere_pm25.png")


In [ ]:
# === A5 : corrélations | A6 : états de qualité d'air ===
keyvars = [v for v in ["co2","co","no2","so2","o3","tvoc","pm1","pm2_5","pm10",
                       "temperature","humidity","pressure","oxygen","sound"] if v in lab.columns]
corr = lab[keyvars].corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(keyvars))); ax.set_xticklabels(keyvars, rotation=90)
ax.set_yticks(range(len(keyvars))); ax.set_yticklabels(keyvars)
for i in range(len(keyvars)):
    for j in range(len(keyvars)):
        ax.text(j, i, f"{corr.iloc[i,j]:.1f}", ha="center", va="center", fontsize=7,
                color="black" if abs(corr.iloc[i,j])<0.6 else "white")
ax.set_title("Corrélations entre variables clés (laboratoire)"); ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, label="Pearson r")
save_show(fig, "A5_correlations.png")

st = pd.read_csv(PROC / "laboratory_states.csv"); levels = cfg["state_levels"]
poll = [v for v in ["co2","no2","pm2_5","tvoc","o3","co"] if v in st.columns]
counts = {lv: [100*st[v].value_counts(normalize=True).get(lv,0.0) for v in poll] for lv in levels}
fig, ax = plt.subplots(figsize=(9, 5)); bottom = np.zeros(len(poll))
cmap = {"Excellent":"#1a9850","Good":"#91cf60","Moderate":"#fee08b","Poor":"#fc8d59","Dangerous":"#d73027"}
for lv in levels:
    ax.bar(poll, counts[lv], bottom=bottom, label=lv, color=cmap.get(lv,"#999")); bottom += np.array(counts[lv])
ax.set_ylabel("% du temps"); ax.set_title("Répartition des états de qualité d'air (laboratoire)")
ax.legend(ncol=5, fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.22))
save_show(fig, "A6_etats_qualite_air.png")


---
## Groupe B — L'ontologie (graphe de connaissances)

Le projet **adopte des standards W3C/OGC existants** plutôt que d'inventer une ontologie :
**SOSA/SSN** (capteurs et observations), **QUDT** (unités et grandeurs physiques) et **OWL-Time**.
Chaque variable devient une *propriété observable* reliée à sa grandeur et à son unité.

Ce graphe sémantique **définit la structure du graphe GNN** : les 30 variables deviennent des nœuds,
leurs relations deviennent des arêtes.

In [ ]:
# === B1 : résumé structurel de l'ontologie ===
items = [("Classes", onto_rep["classes"]), ("Propriétés d'objet", onto_rep["object_properties"]),
         ("Propriétés de données", onto_rep["data_properties"]), ("Individus", onto_rep["individuals"]),
         ("Nœuds (variables)", onto_rep["variables_modeled"]),
         ("Arêtes dirigées (GNN)", onto_rep["directed_edges_for_gnn"]),
         ("Restrictions de classe", onto_rep["class_restrictions"]), ("Triplets RDF", onto_rep["rdf_triples"])]
fig, ax = plt.subplots(figsize=(9, 5.2)); ax.axis("off")
ax.set_title("Ontologie IAQ - résumé structurel\n"
             f"(standards : {', '.join(onto_rep['adopted_ontologies'])} ; "
             f"QUDT : {onto_rep['observable_properties_qudt_mapped']})", pad=18)
rows = (len(items)+1)//2
for i,(k,v) in enumerate(items):
    r,c = i % rows, i // rows; x = 0.04 + c*0.5; y = 0.82 - r*0.20
    ax.add_patch(plt.Rectangle((x, y-0.14), 0.42, 0.16, transform=ax.transAxes,
                 facecolor="#2e86c1", alpha=0.12, edgecolor="#2e86c1"))
    ax.text(x+0.02, y-0.02, f"{v:,}".replace(",", " "), transform=ax.transAxes,
            fontsize=20, fontweight="bold", color="#2e86c1", va="top")
    ax.text(x+0.02, y-0.10, k, transform=ax.transAxes, fontsize=10, va="top")
save_show(fig, "B1_ontologie_resume.png")

# B2 : variables par catégorie
from collections import Counter
cc = Counter(cat.values()); order = sorted(cc, key=lambda k: -cc[k])
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.bar(order, [cc[o] for o in order], color=[CAT_COLORS.get(o,"#999") for o in order])
for b,o in zip(bars, order):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1, str(cc[o]), ha="center", fontweight="bold")
ax.set_ylabel("Nombre de variables"); ax.set_title("Variables par catégorie sémantique")
ax.set_xticklabels(order, rotation=25, ha="right")
save_show(fig, "B2_variables_par_categorie.png")


---
## Groupe C — Le graphe GNN

Le graphe consommé par les réseaux de neurones : **30 nœuds** (les variables), **57 arêtes dirigées**
issues des relations ontologiques (`correlatedWith`, `influences`, `indicates`…). Chaque nœud porte
**8 canaux de caractéristiques** par fenêtre temporelle (moyenne, écart-type, min, max, dernière
valeur, pente, étendue, sévérité d'état).

In [ ]:
# === C1 : topologie | C2 : degrés | C3 : relations ===
edges = pd.read_csv(ONTO / "ontology_edges.csv")
G = nx.DiGraph()
for n in gman["node_order"]:
    G.add_node(n, category=cat.get(n, "MeasurementProperty"))
for _, r in edges.iterrows():
    G.add_edge(r["src"], r["dst"], relation=r["relation"])

# C1
fig, ax = plt.subplots(figsize=(12, 9))
pos = nx.spring_layout(G, seed=42, k=0.9, iterations=200)
deg = dict(G.degree())
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, width=1, arrows=True, arrowsize=8,
                       edge_color="#555", connectionstyle="arc3,rad=0.06")
nx.draw_networkx_nodes(G, pos, ax=ax,
                       node_color=[CAT_COLORS.get(G.nodes[n]["category"], "#999") for n in G.nodes],
                       node_size=[200+120*deg[n] for n in G.nodes], edgecolors="white", linewidths=1.2)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7.5)
cats_present = sorted(set(cat.get(n, "MeasurementProperty") for n in G.nodes))
ax.legend(handles=[Patch(facecolor=CAT_COLORS.get(c,"#999"), label=c) for c in cats_present],
          loc="upper left", fontsize=8, title="Catégorie")
ax.set_title(f"Graphe de connaissances IAQ - {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes\n"
             "(taille ∝ degré, couleur = catégorie)"); ax.axis("off")
save_show(fig, "C1_graphe_gnn.png")

# C2
degs = sorted(dict(G.degree()).items(), key=lambda x: -x[1])
names = [d[0] for d in degs]; vals = [d[1] for d in degs]
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(names, vals, color=[CAT_COLORS.get(cat.get(n,""),"#999") for n in names])
ax.set_xticklabels(names, rotation=90, fontsize=7.5); ax.set_ylabel("Degré (nombre de connexions)")
ax.set_title("Degré de chaque nœud du graphe")
save_show(fig, "C2_degres_noeuds.png")

# C3
vc = edges["relation"].value_counts()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(vc.index[::-1], vc.values[::-1], color="#16a085")
for i, v in enumerate(vc.values[::-1]):
    ax.text(v+0.3, i, str(v), va="center", fontweight="bold")
ax.set_xlabel("Nombre d'arêtes"); ax.set_title("Types de relations dans le graphe (vocabulaire ontologique)")
save_show(fig, "C3_types_relations.png")


---
## Groupe D — La détection (cœur du projet)

**Principe.** On entraîne un **auto-encodeur** uniquement sur des fenêtres *normales* : il apprend à
reconstruire le comportement normal. Une fenêtre mal reconstruite (**erreur de reconstruction**
élevée) est considérée comme anormale. Trois modèles sont comparés : **GCN**, **GIN**, **SAT-Graph**.

**Le problème découvert.** Avec un seuil au **99ᵉ percentile** de l'erreur normale, **0/3 événements**
étaient détectés. La figure D1 explique pourquoi : la distribution d'erreur normale a une **queue
extrêmement lourde** (quelques fenêtres normales atypiques atteignent des centaines, alors que la
médiane est ~0,18). Le p99 se retrouve donc *au-dessus* des vrais événements, dans une « zone morte ».

**La solution.** Un **seuil robuste médiane + k·MAD** (écart absolu médian) ignore cette queue.
Résultat (figure D4) : **GCN passe de 0/3 à 3/3 événements détectés**, F1 = 0,345.

**Métriques utilisées** (rappel) :
- **Rappel (recall)** = proportion des vrais événements détectés (la métrique clé ici) ;
- **Précision** = proportion des alertes qui sont de vrais événements ;
- **FPR** = taux de fausses alertes sur les fenêtres normales (à garder bas) ;
- **F1** = compromis entre rappel et précision ;
- **ROC-AUC** = séparabilité globale événements/normal (0,5 = hasard).

In [ ]:
# === D1 : distribution à queue lourde + seuils (FIGURE CENTRALE) ===
Z = np.load(GRAPH / "recon_errors.npz", allow_pickle=True)
det_val = Z["det_val"]; det_test = Z["det_test"]; eid = Z["event_id"]
ev_names = {0:"normal",1:"fire",2:"dust_humidity",3:"power_outage"}
err = Z["GCN"]; val_err = err[det_val]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.hist(val_err, bins=60, color="#2e86c1", alpha=0.7)
ax.axvline(np.percentile(val_err, 99), color="#d73027", ls="--", lw=1.5,
           label=f"Seuil p99 = {np.percentile(val_err,99):.0f}")
ax.set_title("Erreur de reconstruction (normal-val) - échelle linéaire\nla queue extrême tire le p99 très haut")
ax.set_xlabel("Erreur de reconstruction (GCN)"); ax.set_ylabel("Nombre de fenêtres"); ax.legend()

ax = axes[1]
bins = np.logspace(np.log10(err.min()+1e-6), np.log10(err.max()), 60)
ax.hist(val_err, bins=bins, color="#2e86c1", alpha=0.55, label="Normal (val)")
for k in [1,2,3]:
    ax.hist(err[(eid==k)&det_test], bins=bins, alpha=0.6,
            color=EVENT_COLORS[ev_names[k]], label=EVENT_FR[ev_names[k]])
med = np.median(val_err); mad = np.median(np.abs(val_err-med)); thr_mad = med + 4*1.4826*mad
ax.axvline(np.percentile(val_err,99), color="#d73027", ls="--", lw=1.5, label=f"p99 = {np.percentile(val_err,99):.0f}")
ax.axvline(thr_mad, color="#1a9850", lw=2, label=f"médiane+4·MAD = {thr_mad:.2f}")
ax.set_xscale("log")
ax.set_title("Échelle log - les événements sont dans la zone normale ;\nle seuil robuste (MAD) tombe au bon endroit")
ax.set_xlabel("Erreur de reconstruction (GCN, log)"); ax.legend(fontsize=8)
fig.suptitle("Pourquoi le seuil p99 échoue : distribution d'erreur à queue lourde",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout(rect=[0,0,1,0.99])
save_show(fig, "D1_queue_lourde_seuils.png")


In [ ]:
# === D2 : erreur par fenêtre de test (avec les deux seuils) ===
fig, ax = plt.subplots(figsize=(12, 5)); idx = np.arange(len(err))
m0 = det_test & (eid==0)
ax.scatter(idx[m0], err[m0], s=8, color="#bbb", label="Normal (test)")
for k in [1,2,3]:
    m = det_test & (eid==k)
    ax.scatter(idx[m], err[m], s=22, color=EVENT_COLORS[ev_names[k]],
               label=EVENT_FR[ev_names[k]], edgecolors="k", linewidths=0.3)
ax.axhline(med+4*1.4826*mad, color="#1a9850", lw=2, label="Seuil médiane+4·MAD")
ax.axhline(np.percentile(val_err,99), color="#d73027", ls="--", lw=1.5, label="Seuil p99")
ax.set_yscale("log"); ax.set_ylabel("Erreur de reconstruction (GCN, log)"); ax.set_xlabel("Indice de fenêtre")
ax.set_title("Erreur de reconstruction par fenêtre de test (GCN)"); ax.legend(fontsize=8, ncol=2)
save_show(fig, "D2_erreur_par_fenetre.png")


In [ ]:
# === D3 : comparaison des stratégies de seuil par modèle ===
df = pd.read_csv(GRAPH / "detection_robust.csv")
ok = df[df["fpr_normal"] <= 0.10].copy()
best = ok.sort_values("f1", ascending=False).groupby(["model","strategy"]).first().reset_index()
models = ["GCN","GIN","SAT-Graph"]; strategies = sorted(best["strategy"].unique())
palette = {"baseline_percentile":"#95a5a6","robust_MAD":"#1a9850","trimmed_percentile":"#e67e22"}
labelmap = {"baseline_percentile":"Percentile brut","robust_MAD":"MAD robuste","trimmed_percentile":"Percentile tronqué"}
fig, ax = plt.subplots(figsize=(10, 5)); w = 0.8/len(strategies)
for i, s in enumerate(strategies):
    vals = [ (best[(best.model==m)&(best.strategy==s)]["f1"].values[0]
              if len(best[(best.model==m)&(best.strategy==s)]) else 0) for m in models]
    ax.bar(np.arange(len(models))+i*w, vals, w, label=labelmap.get(s,s), color=palette.get(s,"#999"))
ax.set_xticks(np.arange(len(models))+w*(len(strategies)-1)/2); ax.set_xticklabels(models)
ax.set_ylabel("Meilleur F1 (FPR ≤ 0.10)")
ax.set_title("Comparaison des stratégies de seuil par modèle\nle seuil MAD robuste domine ; SAT-Graph reste faible")
ax.legend(title="Stratégie")
save_show(fig, "D3_comparaison_strategies.png")


In [ ]:
# === D4 : l'histoire 0/3 -> 3/3 (GCN, p99 vs MAD_k4) ===
g = df[df["model"]=="GCN"]; p99 = g[g["label"]=="base_p99"].iloc[0]; mad_row = g[g["label"]=="MAD_k4"].iloc[0]
evs = ["recall_fire","recall_dust_humidity","recall_power_outage"]
labels = ["Incendie","Poussière/\nhumidité","Coupure\ncourant"]
fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(len(evs)); w = 0.38
ax.bar(x-w/2, [p99[e] for e in evs], w, label="Seuil p99 (NB04 d'origine)", color="#d73027")
ax.bar(x+w/2, [mad_row[e] for e in evs], w, label="Seuil médiane+4·MAD", color="#1a9850")
for i,e in enumerate(evs):
    ax.text(i-w/2, p99[e]+0.01, f"{p99[e]:.2f}", ha="center", fontsize=9)
    ax.text(i+w/2, mad_row[e]+0.01, f"{mad_row[e]:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Rappel par événement")
ax.set_title("Détection GCN : 0/3 → 3/3 grâce au seuil robuste\n"
             f"p99 : 0 détecté  |  MAD : 3/3 détectés (F1 {mad_row['f1']:.2f})"); ax.legend()
save_show(fig, "D4_baseline_vs_mad.png")


In [ ]:
# === D5 : rappel par événement et par modèle | D6 : rappel vs percentile ===
summ = load_json(GRAPH / "detection_robust_summary.json")["best_per_model"]
evs2 = [("recall_fire","Incendie"),("recall_dust_humidity","Poussière/humidité"),("recall_power_outage","Coupure courant")]
M = np.array([[summ[m][e[0]] for e in evs2] for m in models])
fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(M, cmap="YlGn", vmin=0, vmax=0.5, aspect="auto")
ax.set_xticks(range(len(evs2))); ax.set_xticklabels([e[1] for e in evs2])
ax.set_yticks(range(len(models))); ax.set_yticklabels([f"{m}\n({summ[m]['label']})" for m in models])
for i in range(len(models)):
    for j in range(len(evs2)):
        ax.text(j, i, f"{M[i,j]:.0%}", ha="center", va="center", fontweight="bold",
                color="black" if M[i,j]<0.3 else "white")
ax.set_title("Rappel par événement et par modèle\n(meilleur seuil, FPR ≤ 0.10)"); ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, label="Rappel")
save_show(fig, "D5_rappel_par_evenement.png")

dre = pd.read_csv(GRAPH / "detection_reeval.csv")
cols = {c.lower(): c for c in dre.columns}
pcol = cols.get("op_point") or cols.get("label") or dre.columns[1]; rcol = cols.get("recall"); mcol = cols.get("model")
order = ["p90","p92.5","p95","p97.5","p99"]
fig, ax = plt.subplots(figsize=(9, 5))
for m, color in zip(models, ["#2e86c1","#16a085","#8e44ad"]):
    sub = dre[dre[mcol]==m]; sub = sub[sub[pcol].astype(str).isin(order)]
    sub = sub.set_index(pcol).reindex(order).reset_index()
    ax.plot(sub[pcol], sub[rcol], "o-", color=color, label=m, lw=2)
ax.set_xlabel("Percentile du seuil (sur normal-val)"); ax.set_ylabel("Rappel global")
ax.set_title("Rappel en fonction du percentile de seuil\nau-delà de p95, le seuil entre dans la « zone morte »"); ax.legend()
save_show(fig, "D6_recall_vs_percentile.png")


---
## Groupe E — La prédiction (comparaison contrôlée)

Tâche secondaire : prédire les concentrations de polluants (régression), en comparant les GNN entre
eux puis aux modèles séquentiels **RNN / LSTM / GRU** (la base « Xia »). Le protocole est
**sans fuite de données** : mêmes découpages train/val/test (2886 / 618 / 620), même normaliseur,
et exclusion des entrées qui « tricheraient » (comptes de particules pour prédire la masse PM, etc.).

**Métriques** : **R²** (1 = parfait, 0 = équivaut à prédire la moyenne, négatif = pire), **MAE** et
**RMSE** (erreurs en unités réelles). La figure E2 montre une nuance importante : certains polluants
(**O₃, SO₂, CO₂**) sont prévisibles, d'autres (**PM, CO**) ne le sont pas dans ce cadre honnête.

In [ ]:
# === E1 : barres R²/MAE/RMSE | E2 : R² par polluant ===
dm = pd.read_csv(GRAPH / "nb05_pred_metrics.csv").sort_values("R2_avg", ascending=False)
colors = ["#2e86c1" if f=="GNN" else "#e67e22" for f in dm["Family"]]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax,(col,title) in zip(axes, [("R2_avg","R² moyen"),("MAE_avg","MAE moyen"),("RMSE_avg","RMSE moyen")]):
    bars = ax.bar(dm["Model"], dm[col], color=colors); ax.set_title(title)
    ax.set_xticklabels(dm["Model"], rotation=35, ha="right")
    if col=="R2_avg": ax.axhline(0, color="k", lw=0.8)
    for b,v in zip(bars, dm[col]):
        ax.text(b.get_x()+b.get_width()/2, v, f"{v:.2f}", ha="center",
                va="bottom" if v>=0 else "top", fontsize=8)
fig.legend(handles=[Patch(facecolor="#2e86c1", label="GNN"),
                    Patch(facecolor="#e67e22", label="Séquentiel (RNN/LSTM/GRU)")],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.06))
fig.suptitle("Prédiction de polluants - comparaison GNN vs séquentiel (sans fuite)", y=1.10,
             fontsize=13, fontweight="bold"); fig.tight_layout(rect=[0,0,1,0.98])
save_show(fig, "E1_prediction_barres.png")

dr2 = pd.read_csv(GRAPH / "nb05_pred_r2.csv").set_index("Model")
polls = [c for c in dr2.columns if c != "AVG"]; Mr = dr2[polls].values
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(Mr, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(polls))); ax.set_xticklabels(polls)
ax.set_yticks(range(len(dr2.index))); ax.set_yticklabels(dr2.index)
for i in range(Mr.shape[0]):
    for j in range(Mr.shape[1]):
        ax.text(j, i, f"{Mr[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="black" if abs(Mr[i,j])<0.5 else "white")
ax.set_title("R² par polluant et par modèle\ncertains polluants (O₃, SO₂, CO₂) sont prévisibles ; les PM ne le sont pas")
ax.grid(False); fig.colorbar(im, ax=ax, fraction=0.046, label="R²")
save_show(fig, "E2_prediction_r2_heatmap.png")


---
## Groupe F — Synthèse

Le tableau de bord final rappelle que **détection et prédiction sont deux axes distincts** (jamais
moyennés). Côté détection, le seuil robuste fait passer GCN de 0/3 à 3/3. Côté prédiction, les
modèles séquentiels devancent légèrement les GNN, mais les performances restent modestes — un
résultat *honnête* et attendu dans un cadre sans fuite de données.

In [ ]:
# === F1 : tableau de bord final ===
final = load_json(GRAPH / "nb06_final_summary.json")
robust = load_json(GRAPH / "detection_robust_summary.json")
gcn = robust["best_per_model"]["GCN"]; p = final["PREDICTION (comparison)"]

def panel(ax, title, lines, accent):
    ax.axis("off"); ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.add_patch(plt.Rectangle((0,0), 1, 0.90, transform=ax.transAxes,
                 facecolor=accent, alpha=0.05, edgecolor=accent, lw=1.5))
    ax.text(0.5, 0.955, title, transform=ax.transAxes, ha="center", fontsize=13, fontweight="bold", color=accent)
    n = len(lines)
    for i,(k,v,c) in enumerate(lines):
        y = 0.80 - i*(0.72/max(n,1))
        ax.text(0.05, y, k, transform=ax.transAxes, fontsize=11, va="center")
        ax.text(0.95, y, v, transform=ax.transAxes, fontsize=14, fontweight="bold", color=c, va="center", ha="right")
        ax.plot([0.05,0.95], [y-0.045,y-0.045], transform=ax.transAxes, color="#ddd", lw=0.8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
panel(axes[0], "DÉTECTION  (taux d'événements)", [
    ("Avant - seuil p99", "0 / 3 détectés", "#d73027"),
    ("ROC-AUC", f"{final['DETECTION (headline)']['ROC_AUC']:.3f}", "#7f8c8d"),
    ("Après - GCN + MAD", f"{gcn['events_detected']} détectés", "#1a9850"),
    ("F1 (GCN + MAD)", f"{gcn['f1']:.3f}", "#1a9850"),
    ("Rappel incendie", f"{gcn['recall_fire']:.0%}", "#1a9850"),
    ("FPR normal", f"{gcn['fpr_normal']:.1%}", "#7f8c8d"),
], "#c0392b")
panel(axes[1], "PRÉDICTION  (régression, sans fuite)", [
    ("Meilleur GNN", p["best_GNN"], "#2e86c1"),
    ("R² meilleur GNN", f"{p['best_GNN_R2']:.3f}", "#2e86c1"),
    ("Meilleur séquentiel", p["best_sequential"], "#e67e22"),
    ("R² meilleur séquentiel", f"{p['best_seq_R2']:.3f}", "#e67e22"),
    ("Constat", "séquentiel > GNN", "#2c3e50"),
], "#2471a3")
fig.suptitle("Tableau de bord final - détection ≠ prédiction (axes séparés)",
             fontsize=14, fontweight="bold", y=1.04); fig.tight_layout(rect=[0,0,1,0.99])
save_show(fig, "F1_tableau_de_bord.png")

print("\n=== Toutes les figures ont été générées dans le dossier figures/ ===")
import os
for f in sorted(os.listdir(FIG)):
    print("  -", f)


---
### Fin du Notebook 7

Toutes les figures sont enregistrées dans `figures/` et prêtes pour le rapport. Les chiffres
proviennent **exclusivement** des fichiers de résultats réels du projet ; aucune valeur n'a été
inventée. Les figures à mettre en avant dans le rapport sont :

- **D1** (queue lourde + seuils) et **D4** (0/3 → 3/3) : le cœur de la contribution sur la détection ;
- **C1** (graphe de connaissances) : illustre l'apport de l'ontologie ;
- **E2** (R² par polluant) : nuance la conclusion sur la prédiction ;
- **F1** (tableau de bord) : synthèse en une image.
